# 08 머신러닝 예측 모델 (Phase 1 — 2/3)

**40조건** × **RF, XGBoost** | 지표: MAE, RMSE, MAPE, MASE

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 165
학습 <= 201730 | 검증: [201731, 201732, 201733]
Best 선정 기준: MAPE


### ① ML 모델 실험

In [2]:
# Phase 1 — ML 2종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_ml_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=ML_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=ML_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_ml_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_ml_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))


Phase1 SBC:   0%|          | 0/20 [00:00<?, ?it/s]

C:\Users\kjh\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:751: UserWarning: [12:57:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Phase1 ML:   0%|          | 0/20 [00:00<?, ?it/s]

완료 | 660 rows


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
1,ML,A,1,XGBoost,4944.263212,6839.595340,95.492940,4.187199
3,ML,A,2,XGBoost,138248.541667,180834.099539,115.117649,4.445066
5,ML,A,3,XGBoost,51902.310242,77426.125331,89.685380,2.873995
6,ML,A,4,RF,147896.034269,199772.436484,95.039133,3.141741
8,ML,B,1,RF,3181.590236,4267.287831,102.193719,3.124449
11,ML,B,2,XGBoost,56637.979000,87948.126476,64.711808,1.891557
13,ML,B,3,XGBoost,36039.312500,54848.326416,58.540456,2.606062
14,ML,C,1,RF,3518.191587,4778.459710,112.719407,3.920559
16,ML,C,2,RF,75887.663677,113124.954524,74.104331,3.271271
18,ML,C,3,RF,60352.684444,73655.401918,62.941009,5.101073



=== 알고리즘별 평균 MAPE ===
cluster_scheme      ML     SBC
model                         
RF              114.59  117.59
XGBoost         208.88  112.57
